# Mass and heat transfer

How does heat spread through a plate? The same model also describes diffusion of a dissolved substance and the transport of particles in the flow of a fluid.

## From conservation to a PDE

Let $u(x,t)$ denote temperature or concentration, $q$ its flux, and $s$ the source. Conservation and a constitutive equations give

$$
\partial_t u + \operatorname{div}q=s,
\qquad
q=-a\nabla u+\vec bu,\quad s= f-cu,
$$

where $a$ is the diffusion coefficient, $\vec b$ the flow velocity, $f$ an external source, and $c$ a reaction force.

Inserting the constitutive laws into the balance equation gives

$$
\partial_t u-\operatorname{div}(a\nabla u-\vec bu)+cu=f,\qquad +\text{ bc.}
$$

In this numerical example, we consider only diffusion, i.e., $\partial_tu-a\Delta u=0$. The edge of the plate is held at temperature zero, $u=0$.

In [ ]:
from netgen.occ import OCCGeometry, Rectangle
from ngsolve import Mesh, H1, GridFunction, BilinearForm, Grad, dx, x, y, exp
from ngsolve.webgui import Draw

# plate centered at origin
plate = Rectangle(2, 1).Face().Move((-1, -0.5, 0))
plate.edges.name = "cold_boundary"
mesh = Mesh(OCCGeometry(plate, dim=2).GenerateMesh(maxh=0.15))
Draw(mesh);

Simple implicit Euler time stepping scheme: For given $u^n$ find $u^{n+1}$ such that for all $v$
$$
\begin{align*}
\int_{\Omega}\frac{u^{n+1}-u^n}{\tau}\,v\,dx + \int_{\Omega}a\nabla u^{n+1}\cdot\nabla v\,dx = \int_{\Omega}f^n\,v\,dx
\end{align*}
$$

is of the form
$$
\begin{align*}
M\frac{\underline{u}^{n+1}-\underline{u}^n}{\tau} + A\underline{u}^{n+1} = \underline{f}^n
\end{align*}
$$
such that after rewriting
$$
\begin{align*}
\underline{u}^{n+1} = \underline{u}^n + \tau\left(M+\tau A\right)^{-1}(\underline{f}^n-A\underline{u}^n).
\end{align*}
$$

In [ ]:
# Numerical solver
fes = H1(mesh, order=2, dirichlet="cold_boundary")
u, v = fes.TnT()

diffusivity = 0.08
dt = 0.02
mass = BilinearForm(u * v * dx).Assemble()
stiffness = BilinearForm(diffusivity * Grad(u) * Grad(v) * dx).Assemble()
time_step = BilinearForm((u * v + dt * diffusivity * Grad(u) * Grad(v)) * dx).Assemble()
inverse = time_step.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")

temperature = GridFunction(fes)
initial_temperature = 6 * (1-x*x) * (0.25-y*y) * exp(-60*((x+0.45)**2+y*y))
Draw(initial_temperature, mesh, deformation=True, order=4)
temperature.Set(initial_temperature)

residual = temperature.vec.CreateVector()
history = GridFunction(fes, multidim=0)
history.AddMultiDimComponent(temperature.vec)

# time stepping
for step in range(40):
    # Here f^n = 0, so the residual is f^n - A u^n = -A u^n.
    residual.data = -stiffness.mat * temperature.vec
    temperature.vec.data += dt * inverse * residual
    if (step + 1) % 5 == 0:
        history.AddMultiDimComponent(temperature.vec)

In [ ]:
Draw(history, mesh, "temperature", interpolate_multidim=True, animate=True, deformation=True, order=4);

## Observe

- Why does the hot spot become wider and lower?
- Where can you recognize the boundary condition $u=0$?
- What do you expect if `diffusivity` is doubled?


[← Lecture overview](index.ipynb) · [Next: minimal surfaces →](02_minimal_surface_and_membrane.ipynb)